# Города: программы с фильтром по жанру и времени выхода
Пример выгрузки выходов программ определенного жанра с ограничением по ЦА и и времени начала

## Описание задачи и условий расчета
- Регион: Владивосток
- Период: 01.09.2023 - 30.11.2023
- ЦА: Все 18+
- Место просмотра: дом
- Каналы: все каналы проекта TV Index
- Программы: жанр - кинофильм, Время начала – 15:45-24:45
- Статистики: Rtg%, Share, AvReach%

## Инициализация

При построении отчета первый шаг в любом ноутбуке - загрузка библиотек, которые помогут обращаться к TVI API и работать с данными.

Выполните следующую ячейку, для этого перейдите в нее и нажмите Ctrl+Enter

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
import os
import re
import json
import datetime
import time
import pandas as pd
#import matplotlib.pyplot as plt
from IPython.display import JSON

from mediascope_api.core import net as mscore
from mediascope_api.mediavortex import tasks as cwt
from mediascope_api.mediavortex import catalogs as cwc

# Настраиваем отображение

# Включаем отображение всех колонок
pd.set_option('display.max_columns', None)
# Задаем максимальное количество выводимых строк. Раскомментируйте нужную строку
# 200 строк
# pd.set_option("display.max_rows", 200)
# Отображаем все строки. ВАЖНО! Отображение большого DataFrame требует много ресурсов
# pd.set_option("display.max_rows", None)

# Cоздаем объекты для работы с TVI API
mnet = mscore.MediascopeApiNetwork()
mtask = cwt.MediaVortexTask()
cats = cwc.MediaVortexCats()

## Справочники
Получим идентификаторы, которые будут использоваться для формирования условий расчета

In [ ]:
# В справочнике регионов найдем id региона Владивосток
cats.get_tv_region(name=['Владивосток'])

# Это id 18

In [ ]:
# В справочнике мест просмотра найдем идентификатор для домашнего смотрения
cats.get_tv_location(name=['дом'])

# Это id 1

In [ ]:
#В каталоге жанров программ найдем id жанра Кинофильм
cats.get_tv_program_type(name=['кинофильм'])

# Это id 11

In [ ]:
# В словаре демографических атрибутов найдем переменную Возраст
cats.find_tv_property('Возраст')

# Для формирования демо фильтра нам понадобится значение столбца entityName: age

## Формирование задания
Зададим условия расчета

In [ ]:
# Задаем период
# Период указывается в виде списка ('Начало', 'Конец') в формате 'YYYY-MM-DD'. Можно указать несколько периодов
date_filter = [('2023-09-01', '2023-11-30')]

# Задаем дни недели
weekday_filter = None

# Задаем тип дня
daytype_filter = None

# Задаем ЦА - все 18+
basedemo_filter = 'age >= 18'

# Доп фильтр ЦА, нужен только в случае расчета отношения между ЦА, например, при расчете Affinity Index
targetdemo_filter = None

# Задаем место просмотра: дом
location_filter = 'locationId = 1'

# Задаем каналы: Регион - Владивосток
company_filter = 'regionId = 18'

# Указываем фильтр программ: жанр - Телесериал, время начала с 15:45 до 24:45  
program_filter = 'programTypeId = 11 and programStartTime >= 154500 and programStartTime <= 244500'

# Фильтр блоков
break_filter = None

# Фильтр роликов
ad_filter = None

# Указываем список срезов
slices = [
    'regionName', #Регион
    'programSpotId', #Программа ID выхода, обязательный атрибут! 
    'programName', #Программа
    'tvCompanyName', #Телекомпания
    'researchDate', #Дата, обязательный атрибут! 
    'researchWeekDay', #День недели
    'programStartTime', #Программа время начала
    'programFinishTime', #Программа время окончания
    'programTypeName', #Программа жанр
    'programCategoryName' #Программа категория
]

# Указываем список статистик для расчета
statistics = ['RtgPer', 'Share', 'ReachPer']

# Задаем условия сортировки: дата (от старых к новым), время начала (по возр.)
sortings = {'researchDate':'ASC', 'programStartTime':'ASC'}

# Задаем опции расчета
options = {
    "kitId": 3, #TV Index Cities
    "issueType": "PROGRAM" #Тип события - Программы
}

## Расчет задания

In [ ]:
# Формируем задание для API TV Index в формате JSON
task_json = mtask.build_simple_task(date_filter=date_filter, weekday_filter=weekday_filter, 
                                        daytype_filter=daytype_filter, company_filter=company_filter, 
                                        location_filter=location_filter, basedemo_filter=basedemo_filter, 
                                        targetdemo_filter=targetdemo_filter,program_filter=program_filter, 
                                        break_filter=break_filter, ad_filter=ad_filter, 
                                        slices=slices, statistics=statistics, sortings=sortings, options=options)

# Отправляем задание на расчет и ждем выполнения
task_simple = mtask.wait_task(mtask.send_simple_task(task_json))

# Получаем результат
df = mtask.result2table(mtask.get_result(task_simple))

In [ ]:
# Приводим порядок столбцов в соответствие с условиями расчета
df = df[slices+statistics]
df

## Сохраняем в Excel
По умолчанию файл сохраняется в папку `excel`

In [ ]:
df_info = mtask.task_builder.get_report_info()

with pd.ExcelWriter(mtask.task_builder.get_excel_filename('cities_03_genre_filter')) as writer:
    df.to_excel(writer, sheet_name='Report', index=True)
    df_info.to_excel(writer, sheet_name='Info', index=False)